In [1]:
print("prinom")

prinom


In [18]:
import pandas as pd
import numpy as np
import glob

folder = "/kaggle/input/datasets/prinom/mixed-mode-data-csv"

files = glob.glob(f"{folder}/**/*.csv", recursive=True)
print(files)

df = pd.read_csv(files[0], index_col="Object")

# extra space & spell correction
df = df.apply(lambda s: s.str.strip() if s.dtype == "object" else s)
df["Designation"] = df["Designation"].replace({
    "Asst. Profssor": "Asst. Professor",
    "Assistant Professor": "Asst. Professor",
    "Associate Professor": "Asst. Professor".replace("Asst.", "Asso."),
})

df

['/kaggle/input/datasets/prinom/mixed-mode-data-csv/mixed_mode_data.csv']


,M_Status,Blood_Group,Soft_Drink,Gender,iPhone,Test1,Test2,Credit_Rating,Designation,Height,Weight
Object,,,,,,,,,,,
1,Single,AB+,Coke,M,Y,Y,N,Excellent,Lecturer,56,110
2,Married,A+,Fanta,F,Y,N,P,Fair,Lecturer,45,115
3,Married,B+,Pepsi,M,N,N,P,Excellent,Asst. Professor,63,123
4,Divorced,O-,Pepsi,F,N,N,N,Good,Professor,82,135
5,Widowed,O+,Fanta,F,N,Y,N,Good,Asso. Professor,77,170
6,Single,AB+,Coke,F,Y,N,N,Fair,Professor,68,154
7,Married,B+,Fanta,M,N,N,N,Excellent,Asso. Professor,45,112
8,Single,B-,Fanta,F,N,Y,N,Excellent,Lecturer,53,160
9,Single,AB-,Pepsi,F,N,N,P,Good,Asso. Professor,67,163


In [19]:
nominal   = ["M_Status", "Blood_Group", "Soft_Drink"]
sym_bin   = ["Gender"]
asym_bin  = {"iPhone": "Y", "Test1": "Y", "Test2": "P"}   # positive 
ordinal   = {
    "Credit_Rating": ["Fair", "Good", "Excellent"],
    "Designation":   ["Lecturer", "Asst. Professor", "Asso. Professor", "Professor"],
}
numeric   = ["Height", "Weight"]
n = len(df)

In [20]:
# 1) Nominal ও Symmetric Binary: if match 0, else 1
def nominal_d(x, y):
    if x == y:
        return 0, 1
    return 1, 1

# 2) Asymmetric Binary: if both negative: 0
def asym_binary_d(x, y, pos):
    if x != pos and y != pos:      # both negative
        return 0, 0                # d = 0, delta = 0 
    if x == y:
        return 0, 1
    return 1, 1

# 3) Ordinal: rank -> 0-1 scale -> Ordinal values have an order, so a “Fair” vs “Excellent” gap should count more than “Fair” vs “Good”; plain 0/1 matching would lose that. Scaling ranks to 0-1 (rank / (M-1)) keeps every attribute on the same scale, so no attribute dominates the final distance.
def ordinal_d(x, y, order):
    M = len(order)
    zx = order.index(x) / (M - 1)
    zy = order.index(y) / (M - 1)
    return abs(zx - zy), 1
 
# 4) Numeric: max - min normalization
def numeric_d(x, y, rng):
    return abs(x - y) / rng, 1

In [21]:
def build_matrix(col, func):
    values = df[col].tolist()
    n = len(values)
    D = np.zeros((n, n))      # distance
    W = np.zeros((n, n))      # delta
    for i in range(n):
        for j in range(n):
            D[i, j], W[i, j] = func(values[i], values[j])
    return D, W

In [22]:
D, W = {}, {}

for c in nominal + sym_bin:
    D[c], W[c] = build_matrix(c, nominal_d)

for c, pos in asym_bin.items():
    D[c], W[c] = build_matrix(c, lambda x, y: asym_binary_d(x, y, pos))

for c, order in ordinal.items():
    D[c], W[c] = build_matrix(c, lambda x, y: ordinal_d(x, y, order))

for c in numeric:
    rng = df[c].max() - df[c].min()
    D[c], W[c] = build_matrix(c, lambda x, y: numeric_d(x, y, rng))

def show(c):
    return pd.DataFrame(D[c], index=df.index, columns=df.index).round(3)

In [23]:
for c in D:
    print(c)
    display(show(c))

M_Status


Object,1,2,3,4,5,6,7,8,9,10
Object,,,,,,,,,,
1,0.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0
2,1.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,0.0
3,1.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,0.0
4,1.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0
5,1.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0
6,0.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0
7,1.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,0.0
8,0.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0
9,0.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0


Blood_Group


Object,1,2,3,4,5,6,7,8,9,10
Object,,,,,,,,,,
1,0.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0
2,1.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
3,1.0,1.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0
4,1.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0
5,1.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0
6,0.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0
7,1.0,1.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0
8,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0
9,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0


Soft_Drink


...

Object,1,2,3,4,5,6,7,8,9,10
Object,,,,,,,,,,
1,0.000,0.000,0.333,1.000,0.667,1.000,0.667,0.000,0.667,0.333
2,0.000,0.000,0.333,1.000,0.667,1.000,0.667,0.000,0.667,0.333
3,0.333,0.333,0.000,0.667,0.333,0.667,0.333,0.333,0.333,0.000
4,1.000,1.000,0.667,0.000,0.333,0.000,0.333,1.000,0.333,0.667
5,0.667,0.667,0.333,0.333,0.000,0.333,0.000,0.667,0.000,0.333
6,1.000,1.000,0.667,0.000,0.333,0.000,0.333,1.000,0.333,0.667
7,0.667,0.667,0.333,0.333,0.000,0.333,0.000,0.667,0.000,0.333
8,0.000,0.000,0.333,1.000,0.667,1.000,0.667,0.000,0.667,0.333
9,0.667,0.667,0.333,0.333,0.000,0.333,0.000,0.667,0.000,0.333


Height


Object,1,2,3,4,5,6,7,8,9,10
Object,,,,,,,,,,
1,0.000,0.297,0.189,0.703,0.568,0.324,0.297,0.081,0.297,0.459
2,0.297,0.000,0.486,1.000,0.865,0.622,0.000,0.216,0.595,0.757
3,0.189,0.486,0.000,0.514,0.378,0.135,0.486,0.270,0.108,0.270
4,0.703,1.000,0.514,0.000,0.135,0.378,1.000,0.784,0.405,0.243
5,0.568,0.865,0.378,0.135,0.000,0.243,0.865,0.649,0.270,0.108
6,0.324,0.622,0.135,0.378,0.243,0.000,0.622,0.405,0.027,0.135
7,0.297,0.000,0.486,1.000,0.865,0.622,0.000,0.216,0.595,0.757
8,0.081,0.216,0.270,0.784,0.649,0.405,0.216,0.000,0.378,0.541
9,0.297,0.595,0.108,0.405,0.270,0.027,0.595,0.378,0.000,0.162


Weight


Object,1,2,3,4,5,6,7,8,9,10
Object,,,,,,,,,,
1,0.000,0.083,0.217,0.417,1.000,0.733,0.033,0.833,0.883,0.117
2,0.083,0.000,0.133,0.333,0.917,0.650,0.050,0.750,0.800,0.033
3,0.217,0.133,0.000,0.200,0.783,0.517,0.183,0.617,0.667,0.100
4,0.417,0.333,0.200,0.000,0.583,0.317,0.383,0.417,0.467,0.300
5,1.000,0.917,0.783,0.583,0.000,0.267,0.967,0.167,0.117,0.883
6,0.733,0.650,0.517,0.317,0.267,0.000,0.700,0.100,0.150,0.617
7,0.033,0.050,0.183,0.383,0.967,0.700,0.000,0.800,0.850,0.083
8,0.833,0.750,0.617,0.417,0.167,0.100,0.800,0.000,0.050,0.717
9,0.883,0.800,0.667,0.467,0.117,0.150,0.850,0.050,0.000,0.767


# Mixed Mode Dissimilarity Matrix

In [24]:
num = sum(W[c] * D[c] for c in D)     # Σ δ·d
den = sum(W.values())                 # Σ δ

d_mixed = pd.DataFrame(num / den, index=df.index, columns=df.index)
d_mixed.round(3)

Object,1,2,3,4,5,6,7,8,9,10
Object,,,,,,,,,,
1,0.000,0.671,0.613,0.862,0.773,0.506,0.600,0.491,0.759,0.537
2,0.671,0.000,0.595,0.783,0.723,0.627,0.572,0.633,0.656,0.557
3,0.613,0.595,0.000,0.653,0.800,0.832,0.334,0.722,0.512,0.437
4,0.862,0.783,0.653,0.000,0.561,0.577,0.777,0.744,0.467,0.771
5,0.773,0.723,0.800,0.561,0.000,0.634,0.704,0.442,0.539,0.682
6,0.506,0.627,0.832,0.577,0.634,0.000,0.851,0.651,0.501,0.674
7,0.600,0.572,0.334,0.777,0.704,0.851,0.000,0.631,0.772,0.617
8,0.491,0.633,0.722,0.744,0.442,0.651,0.631,0.000,0.560,0.759
9,0.759,0.656,0.512,0.467,0.539,0.501,0.772,0.560,0.000,0.676


In [25]:
mask = np.tril(np.ones((n, n), dtype=bool))
d_mixed.where(mask).round(3)

Object,1,2,3,4,5,6,7,8,9,10
Object,,,,,,,,,,
1,0.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0.671,0.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0.613,0.595,0.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0.862,0.783,0.653,0.000,NaN,NaN,NaN,NaN,NaN,NaN
5,0.773,0.723,0.800,0.561,0.000,NaN,NaN,NaN,NaN,NaN
6,0.506,0.627,0.832,0.577,0.634,0.000,NaN,NaN,NaN,NaN
7,0.600,0.572,0.334,0.777,0.704,0.851,0.000,NaN,NaN,NaN
8,0.491,0.633,0.722,0.744,0.442,0.651,0.631,0.000,NaN,NaN
9,0.759,0.656,0.512,0.467,0.539,0.501,0.772,0.560,0.000,NaN


In [26]:
pairs = d_mixed.where(np.tril(np.ones((n, n), dtype=bool), k=-1)).stack()
print("Most similar   :", pairs.idxmin(), round(pairs.min(), 4))
print("Most dissimilar:", pairs.idxmax(), round(pairs.max(), 4))

Most similar   : (np.int64(7), np.int64(3)) 0.3337
Most dissimilar: (np.int64(4), np.int64(1)) 0.8619


**Color**

In [30]:
def hl(v):
    if np.isclose(v, pairs.min()): return "background-color:#2ecc71"
    if np.isclose(v, pairs.max()): return "background-color:#e74c3c"
    return ""

d_mixed.round(3).style.map(hl)     # পুরনো pandas হলে .applymap

Object,1,2,3,4,5,6,7,8,9,10
Object,,,,,,,,,,
1,0.000000,0.671000,0.613000,0.862000,0.773000,0.506000,0.600000,0.491000,0.759000,0.537000
2,0.671000,0.000000,0.595000,0.783000,0.723000,0.627000,0.572000,0.633000,0.656000,0.557000
3,0.613000,0.595000,0.000000,0.653000,0.800000,0.832000,0.334000,0.722000,0.512000,0.437000
4,0.862000,0.783000,0.653000,0.000000,0.561000,0.577000,0.777000,0.744000,0.467000,0.771000
5,0.773000,0.723000,0.800000,0.561000,0.000000,0.634000,0.704000,0.442000,0.539000,0.682000
6,0.506000,0.627000,0.832000,0.577000,0.634000,0.000000,0.851000,0.651000,0.501000,0.674000
7,0.600000,0.572000,0.334000,0.777000,0.704000,0.851000,0.000000,0.631000,0.772000,0.617000
8,0.491000,0.633000,0.722000,0.744000,0.442000,0.651000,0.631000,0.000000,0.560000,0.759000
9,0.759000,0.656000,0.512000,0.467000,0.539000,0.501000,0.772000,0.560000,0.000000,0.676000


In [31]:
d_low = d_mixed.where(np.tril(np.ones((n, n), dtype=bool)))
d_low.style.map(hl).format("{:.3f}", na_rep="")

Object,1,2,3,4,5,6,7,8,9,10
Object,,,,,,,,,,
1,0.000,,,,,,,,,
2,0.671,0.000,,,,,,,,
3,0.613,0.595,0.000,,,,,,,
4,0.862,0.783,0.653,0.000,,,,,,
5,0.773,0.723,0.800,0.561,0.000,,,,,
6,0.506,0.627,0.832,0.577,0.634,0.000,,,,
7,0.600,0.572,0.334,0.777,0.704,0.851,0.000,,,
8,0.491,0.633,0.722,0.744,0.442,0.651,0.631,0.000,,
9,0.759,0.656,0.512,0.467,0.539,0.501,0.772,0.560,0.000,


> **RESULT:**

0.862 means Objects 1 and 4 are the most dissimilar pair, since they differ on almost every attribute. 

0.334 means Objects 7 and 3 are the most similar pair, since most of their attributes match.